In [1]:
# --- [CELL 0]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 1}
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import pandas as pd

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('data/'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

data/Orbits.csv
data/allStar-dr17-synspec_rev1.fits


In [2]:
# --- [CELL 1]: ---
# cell_state: edited
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 2}
# === BEFORE (original) ===
# filename = "data/allStar-dr17-synspec_rev1.fits"
# data = fits.open(filename)[1].data
# print("Loaded Fits")
# features = []
# # for i in data.columns:
# #     if data[i.name].shape == (733901,):
# #         features.append(i.name)
# 
# 
# print("Loaded Data")
# 
# 
# IDS = []
# 
# toUse = []
# flags = [
#     'APOGEE_ID', 'APOGEE2_TARGET2'
# ]
# features = toUse + flags
# 
# 
# df = pd.DataFrame()
# features=[] + IDS + features
# 
# features = list(set(features))
# for feature in features:
#     print(f"Adding {feature}...", end=" ")
#     df[feature] = data[feature]
#     print("Done")
# 
# # Handle endia\nness issues
# for col in df.columns:
#     if hasattr(df[col], 'dtype') and hasattr(df[col].dtype, 'byteorder'):
#         if df[col].dtype.byteorder == '>':
#             print(f"Converting {col} from big-endian to little-endian")
#             df[col] = df[col].values.astype(df[col].dtype.newbyteorder('<'))
# 
# # data = df
# 
# print("DONE")

# === AFTER (edited) ===
filename = "data/allStar-dr17-synspec_rev1.fits"
data = fits.open(filename)[1].data
print("Loaded Fits")
features = []





print("Loaded Data")


IDS = []

toUse = []
flags = [
    'APOGEE_ID', 'APOGEE2_TARGET2'
]
features = toUse + flags


df = pd.DataFrame()
features=[] + IDS + features

features = list(set(features))
for feature in features:
    print(f"Adding {feature}...", end=" ")
    df[feature] = data[feature]
    print("Done")


for col in df.columns:
    if hasattr(df[col], 'dtype') and hasattr(df[col].dtype, 'byteorder'):
        if df[col].dtype.byteorder == '>':
            print(f"Converting {col} from big-endian to little-endian")
            df[col] = df[col].values.astype(df[col].dtype.newbyteorder('<'))
        elif df[col].dtype.kind == 'O':
            # Also convert object dtype columns to ensure compatibility
            print(f"Converting {col} to native encoding")
            df[col] = df[col].astype(str)


print("DONE")

Loaded Fits
Loaded Data
Adding APOGEE_ID... Done
Adding APOGEE2_TARGET2... Done
Converting APOGEE_ID to native encoding
Converting APOGEE2_TARGET2 from big-endian to little-endian
DONE


In [3]:
# --- [CELL 2]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 3}
data4 = pd.Series(data["APOGEE2_TARGET2"])

In [4]:
# --- [CELL 3]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 4}
IDS = []

toUse = []
flags = [
    'APOGEE_ID', 'APOGEE2_TARGET2'
]
features = toUse + flags


df = pd.DataFrame()
features=[] + IDS + features

features = list(set(features))

for feature in features:
    print(f"Adding {feature}...", end=" ")
    df[feature] = data[feature]
    print("Done")

Adding APOGEE_ID... Done
Adding APOGEE2_TARGET2... Done


In [5]:
# --- [CELL 4]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 5}
data2 = pd.read_csv("data/Orbits.csv")

In [6]:
# --- [CELL 5]: ---
# cell_state: unchanged
# execution_status: {'status': 'error', 'done': True, 'execution_count': 6}
data3 = df.merge(data2, on='APOGEE_ID', how='right')

ValueError: Big-endian buffer not supported on little-endian compiler

In [7]:
import numpy as np

assert 'data3' in globals(), "Merge output `data3` should exist; merge likely failed."
assert isinstance(data3, pd.DataFrame), "`data3` must be a pandas DataFrame after merge."
assert 'APOGEE2_TARGET2' in data3.columns, "Merged data should retain APOGEE2_TARGET2 for downstream bitmask logic."

merged_bo = getattr(data3['APOGEE2_TARGET2'].to_numpy(copy=False).dtype, 'byteorder', '=')
assert merged_bo in ('<', '=', '|'), f"merged APOGEE2_TARGET2 should be native/little-endian: {merged_bo!r}"

halo_bits = data3['APOGEE2_TARGET2'].fillna(0).astype(np.int64) & 524288
assert halo_bits.notna().all(), "Bitmask computation on merged APOGEE2_TARGET2 should succeed without endian-related failure."

AssertionError: Merge output `data3` should exist; merge likely failed.